# Estimating Hyperparameters in `quantEM`

Tutorial by Georgios Varnavides (`G.Varnavides@tudelft.nl`), TU Delft and Stephanie Ribet, LBNL (`sribet@lbl.gov`)

## What this notebook does

The previous notebook handed the reconstruction its aberrations on a plate, because the data was simulated. On a real microscope two sets of numbers are almost never known well enough:

- the **relative rotation** between the detector coordinate system and the scan axes, and
- the **probe aberrations** at the middle plane of the sample.

Both have to come out of the data itself. `quantEM` offers four ways to do that, and this notebook works through all of them on the same dataset so you can see how they compare.

By the end of the notebook you will have:

1. Seen what a reconstruction looks like with the wrong hyperparameters, and learned to read the failure.
2. Run a grid search over a small parameter space.
3. Run a Bayesian search with `optuna` over a larger one.
4. Fitted the aberrations directly, by cross-correlating virtual bright-field images.
5. Fitted them again by least-squares against the aperture overlap functions.

The dataset is small and everything runs on a CPU.

## 1. Set Up the Environment

In [ ]:
%pip install -q git+https://github.com/electronmicroscopy/quantem.git@dev

In [ ]:
import numpy as np
import quantem as em

from quantem.diffractive_imaging.direct_ptychography import OptimizationParameter

## 2. Download and Load the Data

A simulated 4D-STEM scan of a small object on a graphene substrate, recorded at 80 keV with a 20 mrad convergence semi-angle and a large 500 Å defocus, on a 4 Å scan step.

The defocus is deliberate. A strongly defocused probe spreads each scan position over many object pixels, which is what makes the aberration estimate a well-posed problem, and it is also the regime that tilt-corrected bright field and parallax imaging are designed for.

The dataset also carries a **descan**: the direct beam wanders across the detector as the probe scans. Watch for it in the mean pattern below.

In [ ]:
import os
import gdown

dirpath = "/content/"
filepath_data = dirpath + "ducky_20mrad_500A-df_4A-step_cropped.zip"

if not os.path.exists(filepath_data):
    gdown.download(
        id="1tgHq7f5qNi2eDI9YLTD4SSgV6BIfw22o",
        output=filepath_data,
        quiet=False,
    )

In [ ]:
dataset = em.io.load(filepath_data)
dataset

In [ ]:
energy = 80e3
semiangle_cutoff = 20

# the values used in the simulation, which we will pretend not to know.
# note the sign: quantEM's `defocus` keyword is -C10, so an underfocus of
# 500 Angstrom corresponds to C10 = -500.
known_C10 = -500.0
known_rotation_angle = -15.0

Take that sign convention seriously. `defocus` and `C10` differ by a minus sign, and searching the wrong side of zero is the most common way to waste an afternoon on these methods. If a reconstruction refuses to sharpen no matter how you tune the magnitude, flip the sign before you tune anything else.

## 3. Apply a Finite Dose and Look at the Data

In [ ]:
def add_poisson_noise(dataset, electrons_per_area):
    """Draw Poisson counts at a given fluence. `np.inf` returns the noiseless data."""
    if electrons_per_area == np.inf:
        return dataset

    electrons_per_probe = electrons_per_area * dataset.sampling[:2].prod()

    dataset_noisy = dataset.copy()
    dataset_noisy.array = np.random.poisson(dataset.array * electrons_per_probe)
    return dataset_noisy


np.random.seed(2026)
noisy_dataset = add_poisson_noise(dataset, 1e5)

In [ ]:
em.visualization.show_2d(
    [
        noisy_dataset[0, 0].array,
        noisy_dataset[-1, -1].array,
        noisy_dataset.mean((0, 1)),
    ],
    title=["pattern at (0, 0)", "pattern at (-1, -1)", "mean pattern"],
    power=0.5,
    cmap="turbo",
);

Compare the two individual patterns: the bright-field disk sits in a visibly different place in each. Averaged over the whole scan, that wander smears the disk into a rounded square. This is the descan signature we described on the [calibration page](https://curiousbeams.github.io/workshop-20260810-iucr-4dstem/calibration-disk-detection), and it is one more thing the reconstruction has to cope with.

## 4. The Ground Truth, and a Bad Guess

First, the answer we are aiming for. Build a `DirectPtychography` object with the simulated values.

In [ ]:
ground_truth = em.diffractive_imaging.DirectPtychography.from_dataset4d(
    noisy_dataset,
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
    rotation_angle=known_rotation_angle,
    aberration_coefs={"C10": known_C10},
    verbose=False,
)

ground_truth.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

Now throw that away and start from what you would actually have: nothing.

In [ ]:
direct_ptycho = em.diffractive_imaging.DirectPtychography.from_dataset4d(
    noisy_dataset,
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
    verbose=False,
)

direct_ptycho.hyperparameter_state

Even with no rotation angle supplied, one was estimated during initialization from the curl of the center-of-mass signal. On this dataset that estimate happens to be essentially exact, which is a nice outcome but not one to count on: the curl estimate degrades at low dose and on strongly diffracting samples, and a few degrees of rotation error is enough to matter.

The aberrations are still completely unknown. Reconstruct with this uninformed state and see how bad it is:

In [ ]:
direct_ptycho.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

## 5. Grid Search

The simplest search evaluates a grid of candidate values and keeps the best, scoring each by the variance across the aligned bright-field stack. An `OptimizationParameter` declares which quantities to search and over what range.

Grid search scales badly with the number of parameters, so use it when you have one or two and a rough idea of where they lie. We use the `'parallax'` kernel throughout the search because it is the fastest of the five.

In [ ]:
direct_ptycho = direct_ptycho.grid_search_hyperparameters(
    aberration_coefs={"C10": OptimizationParameter(-1000, 0, n_points=11)},
    rotation_angle=OptimizationParameter(-45, 45, n_points=11),
    deconvolution_kernel="parallax",
)

direct_ptycho.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

In [ ]:
direct_ptycho.hyperparameter_state

The individual trials are kept in `_grid_search_results`, which is worth inspecting to see how sharply peaked the loss actually is:

In [ ]:
def tabular_print(tabular_data, n=15):
    headers = list(tabular_data[0][0].keys()) + ["loss"]
    header_row = "".join([f"{h:^15}" for h in headers])

    print(header_row)
    print("-" * len(header_row))

    for entry, loss in sorted(tabular_data, key=lambda x: x[1])[:n]:
        dic = entry.copy()
        dic["loss"] = loss
        print("".join([f"{dic[h]:>15.4e}" for h in headers]))


tabular_print(direct_ptycho._grid_search_results)

## 6. Bayesian Optimization

For more than a couple of parameters, sampling candidates intelligently beats sampling them uniformly. `quantEM` wraps [`optuna`](https://optuna.org) for this. The syntax is nearly identical, except that `OptimizationParameter` no longer needs `n_points` and there is a global `n_trials` instead.

A useful rule of thumb is roughly 50 trials per parameter being optimized.

In [ ]:
direct_ptycho = direct_ptycho.optimize_hyperparameters(
    aberration_coefs={
        "C10": OptimizationParameter(-1000, 0),
        "C12": OptimizationParameter(0, 200),
        "phi12": OptimizationParameter(-np.pi / 2, np.pi / 2),
    },
    rotation_angle=OptimizationParameter(-45, 45),
    n_trials=250,
    deconvolution_kernel="parallax",
)

direct_ptycho.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

In [ ]:
direct_ptycho.hyperparameter_state

Compare the recovered defocus and rotation against the simulated values at the top of the notebook. Note also that two-fold astigmatism was included in the search even though the simulation has none: a good optimizer should return it close to zero, and if it does not, that is a sign the loss surface is flatter than you would like.

`reconstruct` uses the optimized state by default. To go back to the pre-optimization values, pass `use_initial_state=True`.

## 7. Least-Squares Fitting from Bright-Field Shifts

Searching is not the only option. Because a defocused probe makes each virtual bright-field image a slightly shifted view of the same object, we can *measure* those shifts by cross-correlation and fit them against the gradient of an aberration surface. This is the tilt-corrected bright-field idea from the [parallax page](https://curiousbeams.github.io/workshop-20260810-iucr-4dstem/parallax), used here as a calibration.

This route reaches `C10`, `C12`, `phi12` and the rotation angle, and no others, but it is fast and it does not need a search range.

Cross-correlation needs a reference image, which comes from a reconstruction with the current state. We deliberately reset the guess to zero to show that it can find the answer from scratch. `bin_factors` controls a sequence of refinement rounds: binning the virtual bright-field images first raises their signal-to-noise, and later rounds refine against a progressively better reference.

In [ ]:
direct_ptycho = direct_ptycho.fit_hyperparameters_cross_correlation(
    alignment_method="reference",
    bin_factors=(3, 2, 1, 1, 1),

    # reset to zero, to prove the fit finds these from scratch
    aberration_coefs={"C10": 0.0, "C12": 0.0, "phi12": 0.0},
    rotation_angle=0.0,
)

direct_ptycho.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

In [ ]:
direct_ptycho.hyperparameter_state

There is also `alignment_method='pairwise'`, which builds a self-consistent linear system from pairwise cross-correlations and needs no reference at all. It is elegant, and it works well here, but it is less battle-tested, so treat it as experimental.

## 8. Least-Squares Fitting from Aperture Overlaps

The last method fits the measured aperture overlap ("gamma") functions directly against a basis of aberration functions. Unlike cross-correlation it is not restricted to low orders.

Two knobs matter:

- `num_q_modes` sets how many spatial frequencies enter the least-squares system.
- `q_signal_weight` sets how those frequencies are chosen. Push it towards 0 for Nyquist-sampled data, to prioritize frequencies near the aperture cutoff; use a larger value, around 10, for sub-sampled data, to prioritize frequencies where the object actually has power.

Start with a single solve for defocus alone.

In [ ]:
direct_ptycho = direct_ptycho.fit_hyperparameters_least_squares(
    aberration_coefs={"C10": 0.0, "C12": 0.0, "phi12": 0.0},   # no prior guess
    cartesian_basis=["C10"],
    num_q_modes=12,
    q_signal_weight=10.0,
    verbose=True,
)

direct_ptycho.hyperparameter_state

That undershoots the defocus badly. The fix is to apply the method *recursively*: deconvolve the current estimate from the data and fit only the residual. Passing a nested list asks for one round per inner list.

In [ ]:
direct_ptycho = direct_ptycho.fit_hyperparameters_least_squares(
    aberration_coefs={"C10": 0.0, "C12": 0.0, "phi12": 0.0},   # no prior guess
    cartesian_basis=[["C10"], ["C10"], ["C10"]],
    num_q_modes=12,
    q_signal_weight=10.0,
    verbose=True,
)

direct_ptycho.reconstruct(
    deconvolution_kernel="parallax",
    upsampling_factor=2,
    verbose=False,
).visualize(show_obj_fft=False)

In [ ]:
direct_ptycho.hyperparameter_state

Much closer, though still short of what the cross-correlation fit managed on this dataset. That is worth sitting with rather than glossing over: a strongly defocused, coarsely sampled scan is a hard case for aperture-overlap fitting, and no single estimator wins everywhere.

It is also instructive to fit a parameter the data does not contain. Append `["C10", "C12_a", "C12_b"]` as a fourth round and watch the defocus estimate get *worse*, while a large astigmatism is invented out of noise. Fitting parameters your data does not constrain is not free.

For higher-order aberrations, `cartesian_basis="low_order"` expands to `["C10", "C12_a", "C12_b", "C21_a", "C21_b", "C30"]`, and `fit_method` controls the order in which they are solved:

- `"global"` fits everything in a single solve.
- `"recursive"` fits in cumulative rounds of increasing radial order.
- `"sequential"` fits in exclusive rounds of increasing radial order.

## 9. What to Notice and What to Try Next

Four methods, and no single winner:

| method | reaches | needs | speed |
| --- | --- | --- | --- |
| grid search | anything | a range, and few parameters | slow, scales badly |
| Bayesian (`optuna`) | anything | a range | moderate |
| cross-correlation | `C10`, `C12`, `phi12`, rotation | nothing | fast |
| aperture overlap | any order | nothing | fast |

In practice they compose well. A common workflow is a coarse Bayesian search to get into the right basin, then a least-squares fit to polish, exactly as the [experimental gold nanoparticle notebook](https://githubtocolab.com/curiousbeams/workshop-20260810-iucr-4dstem/blob/main/notebooks/try-it-yourself/quantem_03_ptycho_experimental_workflow.ipynb) does on real data.

Follow-up exercises:

1. Drop `n_trials` to 50 and see whether the Bayesian search still finds the right basin.
2. Widen the `C10` range to include negative values. Does the sign convention behave as you expect?
3. Reduce the dose to 10$^3$ e$^-$/Å$^2$ and see which estimator degrades first.
4. Run the cross-correlation fit with `bin_factors=(1,)` and compare against the multi-round version above.
5. Feed the fitted aberrations into a different deconvolution kernel from the [kernels notebook](https://githubtocolab.com/curiousbeams/workshop-20260810-iucr-4dstem/blob/main/notebooks/try-it-yourself/quantem_01_direct_ptychography_kernels.ipynb).

## References

[1] G. Varnavides, W. P. M. de Kleijne, and S. M. Ribet, "The ABCs of phase retrieval: Connecting the acronyms of scanning transmission electron microscopy," *MRS Bulletin* (2026). DOI: <https://doi.org/10.1557/s43577-026-01100-3>.

[2] `quantEM` documentation: <https://electronmicroscopy.github.io/quantem-docs/>